# NB_Score_Stations
Calcule les indicateurs par station, écrit `station_kpi` et renvoie un JSON au pipeline.

In [ ]:
threshold = 0.10
run_id = "manuel"

In [ ]:
import json
from pyspark.sql import functions as F

s = spark.read.table("sessions")
k = (s.groupBy("station")
      .agg(F.count("*").alias("sessions"),
           F.sum(F.when(F.col("status") == "failed", 1).otherwise(0)).alias("failed"),
           F.round(F.sum("energy_wh") / 1000, 1).alias("energy_kwh"),
           F.round(F.sum("price_cents") / 100, 2).alias("revenue_eur"),
           F.round(F.avg("duration_min"), 1).alias("avg_minutes"))
      .withColumn("failure_rate", F.round(F.col("failed") / F.col("sessions"), 3))
      .withColumn("alert", F.col("failure_rate") > F.lit(float(threshold)))
      .withColumn("run_id", F.lit(run_id))
      .withColumn("scored_at", F.current_timestamp()))
k.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("station_kpi")

rows = k.collect()
worst = max(rows, key=lambda r: r["failure_rate"])
result = {"sessions": sum(r["sessions"] for r in rows),
          "energy_kwh": round(sum(r["energy_kwh"] for r in rows)),
          "alerts": sum(1 for r in rows if r["alert"]),
          "worst": worst["station"],
          "worst_pct": round(worst["failure_rate"] * 100, 1)}
print(result)
notebookutils.notebook.exit(json.dumps(result))